# Path B - Module 19: Structural Repair & Mutagenesis

To make PETase work in an industrial bioreactor, we need to increase its stability. A common engineering trick is to introduce a **Disulfide Bridge** (S-S) between two residues that are close in space but distant in the sequence.

In this module, you will repair the experimental PETase model and perform a **Double Mutation** to create a thermostable variant.

In [ ]:
import molsysmt as msm
from molsysmt import systems

# Load the clean enzyme
molsys = msm.convert('pdb:6EQE', to_form='molsysmt.MolSys', selection='molecule_type=="protein"')

### 1. Reconstructing Sidechains
First, we fix the broken residues we found in the audit module. We use `add_missing_heavy_atoms()` to reconstruct the sidechains using standard templates.

In [ ]:
# Repair the protein structure
msm.build.add_missing_heavy_atoms(molsys)

# Add missing hydrogens for pH 8.0 (industrial conditions)
msm.build.add_missing_hydrogens(molsys, pH=8.0)

print(f"Enzyme repaired. Final atoms: {msm.get(molsys, element='system', n_atoms=True)}")

### 2. Engineering Termostability: The Double Mutation
We want to introduce a disulfide bridge between positions **238** and **289**. For this, we must mutate both residues (Serine and Alanine) to **Cysteines (CYS)**.

In [ ]:
# Identify residue indices for residue IDs 238 and 289
res_indices = msm.select(molsys, selection='group_id==[238, 289]', element='group')

# Perform the double mutation to Cysteine
msm.build.mutate(molsys, selection='group_index=='+str(res_indices[0]), new_group_name='CYS')
msm.build.mutate(molsys, selection='group_index=='+str(res_indices[1]), new_group_name='CYS')

# Verify that both are now Cysteines
names = msm.get(molsys, element='group', selection=res_indices, group_name=True)
print(f"Mutated residues: {names}")

### 3. Adding the Covalent Cross-link
Mutating to Cysteine is not enough; we need to tell the framework that these two atoms are now connected by a disulfide bond.

In [ ]:
# MolSysMT can add explicit bonds between atoms
# Find the Sulfur Gamma (SG) atoms of our new Cysteines
sg_atoms = msm.select(molsys, selection='group_index=='+str(res_indices)+' and atom_name=="SG"')

# We will see how to add these bonds permanently in the Builder module,
# but for now, the mutation engine has placed them in the right position.

--- 

### 🏆 Path B Challenge: The Thermal Optimizer

1. Take your repaired `molsys` enzyme.
2. Mutate **Isoleucine 179** to a **Phenylalanine (PHE)**. This mutation (I179F) is known to improve the hydrophobic packing near the active site.
3. Use `msm.info(element='group', selection='group_id==179')` to verify the change.
4. Check for **Overlapping Molecules** using `msm.build.remove_overlapping_molecules()` to ensure your new PHE doesn't clash with the surroundings.

You have successfully engineered a mutant protein! In **Module 20**, we will learn how to **synthesize** a small PET fragment to see how it fits in your new active site.